In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

# Register font file
font_manager.fontManager.addfont('/content/ARIALBD.TTF')

# Tell matplotlib to use it
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.labelweight'] = 'bold'

In [ ]:
import matplotlib.font_manager as fm
print(fm.findfont('Arial', fallback_to_default=False))

In [ ]:
# --------------------------------------------
# GLOBAL TERMINAL GROUP COLOR DICTIONARY
# --------------------------------------------

terminal_color_map = {
    'O'   : '#E41A1C',  # bright red
    'Cl'  : '#377EB8',  # strong blue
    'Br'  : '#4DAF4A',  # vivid green
    'CN'  : '#984EA3',  # purple
    'PO'  : '#FF7F00',  # orange
    'NCO' : '#FFFF33',  # yellow
    'NCS' : '#A65628',  # brown
    'OCN' : '#F781BF',  # pink
    'SCN' : '#17BECF'   # cyan
}

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

#plt.rcParams['font.family'] = 'Arial'

def parity_plot(x_train, x_test, dataset_name):

    plt.rcParams["font.weight"] = "bold"
    plt.rcParams["axes.labelweight"] = "bold"
    plt.rcParams["figure.dpi"] = 400

    fig, ax = plt.subplots(figsize=(10, 9))

    # Use a consistent terminal group order from the global color map
    ordered_terminals = [t for t in terminal_color_map if
                         t in x_train['T1'].values or t in x_test['T1'].values]

    # ===============================
    # TRAIN DATA
    # ===============================
    for t in ordered_terminals:
        subset = x_train[x_train['T1'] == t]
        if subset.empty:
            continue
        color = terminal_color_map.get(t, 'black')
        ax.scatter(subset['Actual delE'],
                   subset['Predicted delE'],
                   color=color,
                   marker='o',
                   s=180,
                   alpha=0.9,
                   edgecolor='black')

    # ===============================
    # TEST DATA
    # ===============================
    for t in ordered_terminals:
        subset = x_test[x_test['T1'] == t]
        if subset.empty:
            continue
        color = terminal_color_map.get(t, 'black')
        ax.scatter(subset['Actual delE'],
                   subset['Predicted delE'],
                   color=color,
                   marker='s',
                   s=180,
                   alpha=0.9,
                   edgecolor='black')

    # ===============================
    # Ideal Parity Line
    # ===============================
    all_values = pd.concat([x_train, x_test])
    min_val = all_values['Actual delE'].min()
    max_val = all_values['Actual delE'].max()

    ideal_handle, = ax.plot([min_val, max_val],
                             [min_val, max_val],
                             'k--',
                             linewidth=2,
                             label='Ideal')

    # ===============================
    # Regression Line (Train)
    # ===============================
    slope, intercept = np.polyfit(x_train['Actual delE'],
                                  x_train['Predicted delE'], 1)

    x_line = np.linspace(min_val, max_val, 100)
    y_line = slope * x_line + intercept

    fit_handle, = ax.plot(x_line, y_line,
                           color='green',
                           linestyle='--',
                           linewidth=2,
                           label='Fit')

    # ===============================
    # Labels
    # ===============================
    ax.set_xlabel('DFT Calculated $\Delta$E (eV)', fontsize=30)
    ax.set_ylabel('ML Predicted $\Delta$E (eV)', fontsize=30)

    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18)

    # ===============================
    # Legend 1: Train / Test markers — horizontal, single row
    # ===============================
    train_marker = Line2D([0], [0], marker='o', color='w',
                          markerfacecolor='grey', markeredgecolor='black',
                          markersize=10, label='Train')
    test_marker  = Line2D([0], [0], marker='s', color='w',
                          markerfacecolor='grey', markeredgecolor='black',
                          markersize=10, label='Test')

    marker_legend = ax.legend(handles=[train_marker, test_marker],
                              labels=['Train', 'Test'],
                              fontsize=18,
                              loc='lower right',
                              ncol=2,          # side by side in one row
                              frameon=True,
                              bbox_to_anchor=(1.0, 0.0))

    # ===============================
    # Legend 2: Terminal groups + Ideal + Fit
    # ===============================
    dummy = Patch(visible=False, label='')

    terminal_patches = [
        Patch(facecolor=terminal_color_map[t], edgecolor='black', label=t)
        for t in ordered_terminals
    ]

    final_handles = []
    final_labels  = []

    # Terminal groups: two per row
    for i in range(0, len(terminal_patches), 2):
        final_handles.append(terminal_patches[i])
        final_labels.append(ordered_terminals[i])
        if i + 1 < len(terminal_patches):
            final_handles.append(terminal_patches[i + 1])
            final_labels.append(ordered_terminals[i + 1])
        else:
            final_handles.append(dummy)
            final_labels.append('')

    # Ideal and Fit share one row
    final_handles += [ideal_handle, fit_handle]
    final_labels  += ['Ideal',      'Fit']

    main_legend = ax.legend(final_handles, final_labels,
                            fontsize=18,
                            loc='lower right',
                            ncol=2,
                            frameon=True,
                            bbox_to_anchor=(1.0, 0.08))  # shifted up to sit above marker legend

    # Add marker legend back (adding main_legend removes it automatically)
    ax.add_artist(marker_legend)

    plt.tight_layout()
    plt.savefig(f'Parity_{dataset_name}.png',
                dpi=700,
                bbox_inches="tight",
                transparent=True)
    plt.show()

###Dataset D

In [ ]:
import pandas as pd
df = pd.read_excel('KnownwithfeatureD.xlsx')
redF = ['M1','M2','X','T1','M1W', 'M1IE', 'M1R', 'M1MP', 'M1χ', 'M1CE', 'M1VS', 'M1VD', 'M1VF',
       'M2W', 'M2V', 'M2IE', 'M2R', 'M2MP', 'M2χ', 'M2CE', 'M2VS', 'M2VD',
       'M2VF', 'T1W', 'T1R', 'T1MP', 'T1VP', 'T1-M1', 'M1-X', 'X-M2', 'M2-T2',
       'SA', 'SW', 'M1XΔIE', 'M2XΔIE', 'M1XΔχ', 'M2XΔχ', 'M2TΔχ', 'D']
df0=df[redF]
df1 = df0[df0["D"]>-9]
df1

In [ ]:
from sklearn.model_selection import train_test_split
x=df1.drop(['D'], axis=1)
y=df1['D']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size= 0.2, random_state=42)

In [ ]:
x_train_a = x_train.drop(['M1', 'M2','X','T1'],axis=1)
x_test_a = x_test.drop(['M1', 'M2','X','T1'],axis=1)

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score

In [ ]:
extr = ExtraTreesRegressor(bootstrap=False, ccp_alpha=0.0,
                    max_depth=30, max_leaf_nodes=None,
                    max_samples=None, min_impurity_decrease=0.0,
                    min_samples_leaf=1,
                    min_samples_split=5, min_weight_fraction_leaf=0.0,
                    n_estimators=400, n_jobs=None, oob_score=False,
                    random_state=42, verbose=0, warm_start=False)

extr.fit(x_train_a, y_train)
train_pred = extr.predict(x_train_a)
test_pred = extr.predict(x_test_a)

extr_MAE_train = mean_absolute_error(y_train, train_pred)
extr_MAE_test = mean_absolute_error(y_test, test_pred)
extr_r2_train = r2_score(y_train, train_pred)
extr_r2_test = r2_score(y_test, test_pred)

extr_MAE_train, extr_MAE_test, extr_r2_train, extr_r2_test

In [ ]:
x_train['Actual delE'] = y_train
x_train['Predicted delE'] = train_pred
x_test['Actual delE'] = y_test
x_test['Predicted delE'] = test_pred

In [ ]:
parity_plot(x_train, x_test, "D")

###Dataset C

In [ ]:
import pandas as pd
df = pd.read_excel('KnownwithfeatureC.xlsx')
redF = ['M1','M2','X','T1','M1W', 'M1IE', 'M1R', 'M1MP', 'M1χ', 'M1CE', 'M1VS', 'M1VD', 'M1VF',
       'M2W', 'M2V', 'M2IE', 'M2R', 'M2MP', 'M2χ', 'M2CE', 'M2VS', 'M2VD',
       'M2VF', 'T1W', 'T1R', 'T1MP', 'T1VP', 'T1-M1', 'M1-X', 'X-M2', 'M2-T2',
       'SA', 'SW', 'M1XΔIE', 'M2XΔIE', 'M1XΔχ', 'M2XΔχ', 'M2TΔχ', 'C']
df0=df[redF]
df1 = df0[df0["C"]>-5]
df1

In [ ]:
from sklearn.model_selection import train_test_split
x=df1.drop(['C'], axis=1)
y=df1['C']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size= 0.2, random_state=42)

In [ ]:
x_train_a = x_train.drop(['M1', 'M2','X','T1'],axis=1)
x_test_a = x_test.drop(['M1', 'M2','X','T1'],axis=1)

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score

In [ ]:
extr = ExtraTreesRegressor(bootstrap=False, ccp_alpha=0.0,
                    max_depth=5, max_leaf_nodes=None,
                    max_samples=None, min_impurity_decrease=0.0, min_samples_leaf=2,
                    min_samples_split=2, min_weight_fraction_leaf=0.0,
                    n_estimators=50, n_jobs=None, oob_score=False,
                    random_state=42, verbose=0, warm_start=False)

extr.fit(x_train_a, y_train)
train_pred = extr.predict(x_train_a)
test_pred = extr.predict(x_test_a)

extr_MAE_train = mean_absolute_error(y_train, train_pred)
extr_MAE_test = mean_absolute_error(y_test, test_pred)
extr_r2_train = r2_score(y_train, train_pred)
extr_r2_test = r2_score(y_test, test_pred)

extr_MAE_train, extr_MAE_test, extr_r2_train, extr_r2_test

In [ ]:
x_train['Actual delE'] = y_train
x_train['Predicted delE'] = train_pred
x_test['Actual delE'] = y_test
x_test['Predicted delE'] = test_pred

In [ ]:
parity_plot(x_train, x_test, "C")

###Dataset B

In [ ]:
import pandas as pd
df = pd.read_excel('KnownwithfeatureB.xlsx')
redF = ['M1','M2','X','T1','M1W', 'M1IE', 'M1R', 'M1MP', 'M1χ', 'M1CE', 'M1VS', 'M1VD', 'M1VF',
       'M2W', 'M2V', 'M2IE', 'M2R', 'M2MP', 'M2χ', 'M2CE', 'M2VS', 'M2VD',
       'M2VF', 'T1W', 'T1R', 'T1MP', 'T1VP', 'T1-M1', 'M1-X', 'X-M2', 'M2-T2',
       'SA', 'SW', 'M1XΔIE', 'M2XΔIE', 'M1XΔχ', 'M2XΔχ', 'M2TΔχ', 'B']
df0=df[redF]
df1 = df0[df0["B"]>-6.5]
df1

In [ ]:
from sklearn.model_selection import train_test_split
x=df1.drop(['B'], axis=1)
y=df1['B']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size= 0.2, random_state=42)

In [ ]:
x_train_a = x_train.drop(['M1', 'M2','X','T1'],axis=1)
x_test_a = x_test.drop(['M1', 'M2','X','T1'],axis=1)

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score

In [ ]:
extr = ExtraTreesRegressor(bootstrap=False, ccp_alpha=0.0,
                    max_depth=12, max_leaf_nodes=None,
                    max_samples=None, min_impurity_decrease=0.0,
                    min_samples_leaf=4,
                    min_samples_split=5, min_weight_fraction_leaf=0.0,
                    n_estimators=100, n_jobs=None, oob_score=False,
                    random_state=42, verbose=0, warm_start=False)

extr.fit(x_train_a, y_train)
train_pred = extr.predict(x_train_a)
test_pred = extr.predict(x_test_a)

extr_MAE_train = mean_absolute_error(y_train, train_pred)
extr_MAE_test = mean_absolute_error(y_test, test_pred)
extr_r2_train = r2_score(y_train, train_pred)
extr_r2_test = r2_score(y_test, test_pred)

extr_MAE_train, extr_MAE_test, extr_r2_train, extr_r2_test

In [ ]:
x_train['Actual delE'] = y_train
x_train['Predicted delE'] = train_pred
x_test['Actual delE'] = y_test
x_test['Predicted delE'] = test_pred

In [ ]:
parity_plot(x_train, x_test, "B")

###DataB-XGBR

In [ ]:
import pandas as pd
df = pd.read_excel('KnownwithfeatureB.xlsx')
redF = ['M1','M2','X','T1','M1W', 'M1IE', 'M1R', 'M1MP', 'M1χ', 'M1CE', 'M1VS', 'M1VD', 'M1VF',
       'M2W', 'M2V', 'M2IE', 'M2R', 'M2MP', 'M2χ', 'M2CE', 'M2VS', 'M2VD',
       'M2VF', 'T1W', 'T1R', 'T1MP', 'T1VP', 'T1-M1', 'M1-X', 'X-M2', 'M2-T2',
       'SA', 'SW', 'M1XΔIE', 'M2XΔIE', 'M1XΔχ', 'M2XΔχ', 'M2TΔχ', 'B']
df0=df[redF]
df1 = df0[df0["B"]>-6.5]
df1

In [ ]:
from sklearn.model_selection import train_test_split
x=df1.drop(['B'], axis=1)
y=df1['B']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size= 0.2, random_state=42)

In [ ]:
x_train_a = x_train.drop(['M1', 'M2','X','T1'],axis=1)
x_test_a = x_test.drop(['M1', 'M2','X','T1'],axis=1)

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score

In [ ]:
xgbr = XGBRegressor(base_score=0.5, booster='gbtree', callbacks=None,
             colsample_bylevel=1, colsample_bynode=1,
             colsample_bytree=0.9934180709238716, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None,
             gamma=0.5044088838126443, grow_policy='depthwise',
             importance_type=None, interaction_constraints='',
             learning_rate=0.06198846003947516, max_bin=256,
             max_cat_to_onehot=4, max_delta_step=0, max_depth=4, max_leaves=0,
             min_child_weight=1, monotone_constraints='()',
             n_estimators=693, n_jobs=0, num_parallel_tree=1,
             objective='reg:squarederror', random_state=42,
             reg_alpha=0.1535515799502255)

xgbr.fit(x_train_a, y_train)
train_pred = xgbr.predict(x_train_a)
test_pred = xgbr.predict(x_test_a)

xgbr_MAE_train = mean_absolute_error(y_train, train_pred)
xgbr_MAE_test = mean_absolute_error(y_test, test_pred)
xgbr_r2_train = r2_score(y_train, train_pred)
xgbr_r2_test = r2_score(y_test, test_pred)

xgbr_MAE_train, xgbr_MAE_test, xgbr_r2_train, xgbr_r2_test

In [ ]:
x_train['Actual delE'] = y_train
x_train['Predicted delE'] = train_pred
x_test['Actual delE'] = y_test
x_test['Predicted delE'] = test_pred

In [ ]:
y_train_values = {}

# Iterate over unique terminal groups in the dataset
for t1 in x_train['T1'].unique():
    # Select delE values for the current terminal group
    delE_values = x_train.loc[x_train['T1'] == t1, 'Actual delE'].tolist()

    # Assign delE values to the dictionary
    y_train_values[t1] = delE_values

# Print the ml_test_values dictionary
#y_train_values.keys()

terminalgroup = list(x_train['T1'].unique())
final_y_train_values = dict(zip(terminalgroup, list(y_train_values.values())))
#final_y_train_values


train_pred_values = {}

# Iterate over unique terminal groups in the dataset
for t1 in x_train['T1'].unique():
    # Select delE values for the current terminal group
    delE_values = x_train.loc[x_train['T1'] == t1, 'Predicted delE'].tolist()

    # Assign delE values to the dictionary
    train_pred_values[t1] = delE_values

# Print the ml_test_values dictionary
#train_pred_values.keys()

terminalgroup = list(x_train['T1'].unique())
final_train_pred_values = dict(zip(terminalgroup, list(train_pred_values.values())))
#final_train_pred_values






y_test_values = {}

# Iterate over unique terminal groups in the dataset
for t1 in x_test['T1'].unique():
    # Select delE values for the current terminal group
    delE_values = x_test.loc[x_test['T1'] == t1, 'Actual delE'].tolist()

    # Assign delE values to the dictionary
    y_test_values[t1] = delE_values

# Print the ml_test_values dictionary
#y_test_values.keys()

terminalgroup = list(x_test['T1'].unique())
final_y_test_values = dict(zip(terminalgroup, list(y_test_values.values())))
#final_y_test_values


test_pred_values = {}

# Iterate over unique terminal groups in the dataset
for t1 in x_test['T1'].unique():
    # Select delE values for the current terminal group
    delE_values = x_test.loc[x_test['T1'] == t1, 'Predicted delE'].tolist()

    # Assign delE values to the dictionary
    test_pred_values[t1] = delE_values

# Print the ml_test_values dictionary
#test_pred_values.keys()

terminalgroup = list(x_test['T1'].unique())
final_test_pred_values = dict(zip(terminalgroup, list(test_pred_values.values())))
#final_test_pred_values

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
TG = list(final_y_train_values.keys())

TG_test = list(final_test_pred_values.keys())

regression_line = np.polyfit(y_train, train_pred, 1)
regression_fn = np.poly1d(regression_line)

plt.rcParams["font.weight"] = "bold"
plt.rcParams["axes.labelweight"] = "bold"
#plt.rcParams["figure.figsize"] = [16, ]
plt.rcParams["figure.dpi"] = 400

fig, ax = plt.subplots(figsize=(12, 10))

# Plot true train values
for t in TG_test:
    ax.scatter(final_y_train_values[t], final_train_pred_values[t], label=t + ' (Train)', alpha=0.9, s=200)

# Plot true test values
for t in TG_test:
    ax.scatter(final_y_test_values[t], final_test_pred_values[t], label=t + ' (Test)', alpha=0.9, marker='s', s=200)
# Add labels and title

ax.set_xlabel('DFT Calculated $\Delta$E (eV)', fontsize = 25)
ax.set_ylabel('ML Predicted $\Delta$E (eV)', fontsize = 25)
#ax.set_title('True vs ML Predicted Values')

# Add legend
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
ax.legend(fontsize=14, loc='lower right', ncol=2)
plt.plot(y_train, regression_fn(y_train), color='green', linestyle='--', label='Regression Line')
#plt.plot([1, xy_max], [1, xy_max], 'k--', label='ideal')
#plt.plot(y_train, upper_bound, color='red', alpha=0.6, label='Deviation')
#plt.plot(y_train, lower_bound, color='red', alpha=0.6, label='Deviation')
plt.text(-4.2, -1,"Dataset B (XGBR)", weight='bold', fontsize = 18)
#plt.text(-6.2, -1.2,"Train MAE = 0.28 eV", weight='bold', fontsize = 18)
#plt.text(-6.2, -1.4,"Test MAE = 0.33 eV", weight='bold', fontsize = 18)
#plt.text(-6.2, -1.6, "Train $R^2$ = 0.86", weight='bold', fontsize = 18)
#plt.text(-6.2, -1.8, "Test $R^2$ = 0.77", weight='bold', fontsize = 18)
plt.savefig('Scatter_B-xgbr.png', dpi=700, bbox_inches="tight", transparent=True)
plt.show()
#plt.fill_between(x, train_lower_boundary, train_upper_boundary, color='gray', alpha=0.2, label='Train Boundary')

###Dataset A

In [ ]:
import pandas as pd
df = pd.read_excel('KnownwithfeatureA.xlsx')
redF = ['M1','M2','X','T1','M1W', 'M1IE', 'M1R', 'M1MP', 'M1χ', 'M1CE', 'M1VS', 'M1VD', 'M1VF',
       'M2W', 'M2V', 'M2IE', 'M2R', 'M2MP', 'M2χ', 'M2CE', 'M2VS', 'M2VD',
       'M2VF', 'T1W', 'T1R', 'T1MP', 'T1VP', 'T1-M1', 'M1-X', 'X-M2', 'M2-T2',
       'SA', 'SW', 'M1XΔIE', 'M2XΔIE', 'M1XΔχ', 'M2XΔχ', 'M2TΔχ', 'A']
df0=df[redF]
df1 = df0[df0["A"]>-7]
df1

In [ ]:
from sklearn.model_selection import train_test_split
x=df1.drop(['A'], axis=1)
y=df1['A']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size= 0.2, random_state=42)

In [ ]:
x_train_a = x_train.drop(['M1', 'M2','X','T1'],axis=1)
x_test_a = x_test.drop(['M1', 'M2','X','T1'],axis=1)

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score

In [ ]:
extr = ExtraTreesRegressor(bootstrap=False, ccp_alpha=0.0,
                    max_depth=7, max_leaf_nodes=None,
                    max_samples=None, min_impurity_decrease=0.0,
                    min_samples_leaf=1,
                    min_samples_split=2, min_weight_fraction_leaf=0.0,
                    n_estimators=200, n_jobs=None, oob_score=False,
                    random_state=42, verbose=0, warm_start=False)

extr.fit(x_train_a, y_train)
train_pred = extr.predict(x_train_a)
test_pred = extr.predict(x_test_a)

extr_MAE_train = mean_absolute_error(y_train, train_pred)
extr_MAE_test = mean_absolute_error(y_test, test_pred)
extr_r2_train = r2_score(y_train, train_pred)
extr_r2_test = r2_score(y_test, test_pred)

extr_MAE_train, extr_MAE_test, extr_r2_train, extr_r2_test

In [ ]:
x_train['Actual delE'] = y_train
x_train['Predicted delE'] = train_pred
x_test['Actual delE'] = y_test
x_test['Predicted delE'] = test_pred

In [ ]:
parity_plot(x_train, x_test, "A")